# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets


!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.4 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [ ]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)


Zero-shot: Mixto.


In [ ]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada


Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [ ]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso a paso**

1. **Velocidades conocidas**  
   - Tren 1 (el primero): \(v_1 = 80 \text{ km/h}\)  
   - Tren 2 (el segundo): \(v_2 = 120 \text{ km/h}\)

2. **Retraso del segundo tren**  
   El segundo tren parte **2 h** después que el primero.

3. **Distancia que el primer tren recorre antes de que el segundo salga**  
   \[
   d_{\text{adelanto}} = v_1 \times \text{tiempo de retraso} = 80 \text{ km/h} \times 2 \text{ h} = 160 \text{ km}
   \]

4. **Velocidad relativa (cuánto se acerca el segundo respecto al primero)**  
   \[
   v_{\text{rel}} = v_2 - v_1 = 120 \text{ km/h} - 80 \text{ km/h} = 40 \text{ km/h}
   \]

5. **Tiempo que tarda el segundo tren en alcanzar al primero**  
   \[
   t_{\text{corte}} = \frac{d_{\text{adelanto}}}{v_{\text{rel}}} = \frac{160 \text{ km}}{40 \text{ km/h}} = 4 \text{ h}
   \]

   Este es el tiempo *desde el momento en que el segundo tren sale*.

6. **Tiempo total desde el inicio del primer tren**  
   \[
   t_{\text{total}} = \text{tiempo de retras

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [ ]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación


Lo siento, no dispongo de esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [ ]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np


In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)


Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [ ]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica


No, los productos en oferta no son elegibles para devolución, solo para cambio de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

from sentence_transformers import SentenceTransformer
import numpy as np
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")


Cliente de Groq inicializado correctamente.


In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos_guerra = [
    "Rusia-Ucrania.Durante 2026, la guerra entre Rusia y Ucrania continuó siendo uno de los conflictos más importantes del mundo. Los principales actores estatales fueron el gobierno de Rusia, liderado por Vladimir Putin, y el gobierno de Ucrania, encabezado por Volodímir Zelenski. Ambos países mantuvieron operaciones militares en distintas zonas del frente mientras la comunidad internacional seguía participando mediante apoyo económico, asistencia militar y esfuerzos diplomáticos. El conflicto continuó afectando la seguridad europea y los mercados internacionales.",
    "Israel-Palestina.El conflicto entre Israel y Palestina continuó generando tensiones y episodios de violencia durante 2026. Entre los principales actores se encontraban el gobierno israelí y organizaciones palestinas, incluyendo Hamás. Las diferencias políticas, territoriales y de seguridad siguieron dificultando una solución duradera. La situación mantuvo la atención internacional debido a sus consecuencias humanitarias y a los constantes llamados al diálogo entre las partes involucradas.",
    "Sudán.La guerra civil en Sudán permaneció como una de las crisis humanitarias más graves de 2026. Los principales actores fueron las Fuerzas Armadas Sudanesas, dirigidas por Abdel Fattah al-Burhan, y las Fuerzas de Apoyo Rápido (RSF), lideradas por Mohamed Hamdan Dagalo, conocido como 'Hemedti'. Los enfrentamientos provocaron desplazamientos masivos de población y una creciente necesidad de ayuda internacional, además de afectar la estabilidad de toda la región."
]

embeddings_documentos = modelo_embeddings.encode(documentos_guerra)
print("Embeddings generados:", embeddings_documentos.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos_guerra[indice_mas_similar]

pregunta = "En que guerra esta envuelto Vladimir Putin?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)



Fragmento recuperado: Rusia-Ucrania.Durante 2026, la guerra entre Rusia y Ucrania continuó siendo uno de los conflictos más importantes del mundo. Los principales actores estatales fueron el gobierno de Rusia, liderado por Vladimir Putin, y el gobierno de Ucrania, encabezado por Volodímir Zelenski. Ambos países mantuvieron operaciones militares en distintas zonas del frente mientras la comunidad internacional seguía participando mediante apoyo económico, asistencia militar y esfuerzos diplomáticos. El conflicto continuó afectando la seguridad europea y los mercados internacionales.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [ ]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag

prompt_desconocido = (
    "En que guerra esta envuelto Vladimir Putin en 2026?"
)

respuesta_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(respuesta_sin_rag.choices[0].message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación


Hasta la fecha de mi última actualización (junio 2024), la única guerra en la que **Vladimir Putin** y el gobierno ruso están activamente envueltos es el conflicto con Ucrania, iniciado en febrero de 2022. A pesar de los esfuerzos diplomáticos, las sanciones internacionales y las operaciones militares continuas, la situación sigue siendo de alto riesgo y no se ha encontrado un final definitivo.

### Lo que se sabe sobre el conflicto en 2026

| Aspecto | Situación actual (2024) | Posibles desarrollos hacia 2026 |
|---------|-------------------------|--------------------------------|
| **Término** | En curso, sin acuerdo de paz firmado. | La guerra podría continuar hasta 2026, aunque hay escenarios de negociación que podrían reducir el alcance del conflicto. |
| **Pérdidas y devastación** | Más de 10 000 soldados rusos muertos y cientos de miles de civiles desplazados. | La población civil y la infraestructura en ambos países seguirán sufriendo daños, con efectos de largo plazo en la eco

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [ ]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag

prompt_rag = f"""Responde la pregunta del usuario usando SOLO la informacion que se proporciona sobre la guerra, no uses informacion que no sea proporcionada, no inventes respuestas.
Si no tienes informacion para responder claramente a la pregunta, tambien expresalo.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta corta."""

respuesta_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(respuesta_con_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica


En la guerra entre Rusia y Ucrania.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [ ]:
# Mostrar ambas respuestas para comparar
print("#########################################")
print(f"Respuesta SIN RAG: {respuesta_sin_rag.choices[0].message.content}" )

print("\n")
print("#########################################")
print(f"Respuesta CON RAG: {respuesta_con_rag.choices[0].message.content}" )



#########################################
Respuesta SIN RAG: Hasta la fecha de mi última actualización (junio 2024), la única guerra en la que **Vladimir Putin** y el gobierno ruso están activamente envueltos es el conflicto con Ucrania, iniciado en febrero de 2022. A pesar de los esfuerzos diplomáticos, las sanciones internacionales y las operaciones militares continuas, la situación sigue siendo de alto riesgo y no se ha encontrado un final definitivo.

### Lo que se sabe sobre el conflicto en 2026

| Aspecto | Situación actual (2024) | Posibles desarrollos hacia 2026 |
|---------|-------------------------|--------------------------------|
| **Término** | En curso, sin acuerdo de paz firmado. | La guerra podría continuar hasta 2026, aunque hay escenarios de negociación que podrían reducir el alcance del conflicto. |
| **Pérdidas y devastación** | Más de 10 000 soldados rusos muertos y cientos de miles de civiles desplazados. | La población civil y la infraestructura en ambos países s

**La respuesta con RAG fue más especifica y más clara.**